# Competition workspace

Start with the README checklist: record the metric, ID and target columns, split rationale,
rules, hardware, and dataset version. Keep notebook outputs cleared before committing.

Local setup: `uv sync --locked --extra notebook --extra dev`. Select that environment's
Python kernel. For Kaggle, attach the repository source as a dataset, and set
`KAGGLE_CODE_DIR` below to the directory containing `src/`. Core libraries are usually
preinstalled; verify their versions against your local environment. No downloads happen here.

In [ ]:
import os
import sys
from pathlib import Path

# On Kaggle, set this to the attached repository's directory containing src/.
KAGGLE_CODE_DIR = os.environ.get("KAGGLE_CODE_DIR")
candidates = (
    [Path(KAGGLE_CODE_DIR).expanduser()] if KAGGLE_CODE_DIR else [Path.cwd(), *Path.cwd().parents]
)
REPO_ROOT = next(
    (path.resolve() for path in candidates if (path / "src/paths.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Attach the repository source and set KAGGLE_CODE_DIR.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.paths import find_data_root, output_root  # noqa: E402
from src.utils import seed_everything  # noqa: E402

SEED = 42
seed_everything(SEED)
OUTPUT_ROOT = output_root(REPO_ROOT)
print("Repository:", REPO_ROOT)
print("Outputs:", OUTPUT_ROOT)

## Data

Set an explicit path for reproducibility. Leave `DATA_DIR = None` to use
`KAGGLE_DATA_DIR`, matching competition mounts, or the repository's `data/`.
Ambiguous or incomplete locations fail with an actionable error.
For images, change `REQUIRED` to `("train", "test", "sample_submission.csv")`.
Inspect CSV dtypes explicitly, especially IDs with leading zeros and missing-value markers.

In [ ]:
COMPETITION_NAME = ""  # Kaggle slug
DATA_DIR = None  # e.g. REPO_ROOT / "data" or "/kaggle/input/my-competition"
REQUIRED = ("train.csv", "test.csv", "sample_submission.csv")

DATA_ROOT = find_data_root(COMPETITION_NAME, data_dir=DATA_DIR, root=REPO_ROOT, required=REQUIRED)
print("Data:", DATA_ROOT)

## Validation and modeling

Choose a split before feature engineering. Fit imputation, scaling, encoders,
feature selection, target encoders, and resampling only on each training partition.
Keep related entities together; use chronological validation for future predictions.
Account for time gaps and overlapping labels when the competition needs them.

The optional tabular baseline uses `configs/baseline.json` and saves metrics,
holdout predictions, split row positions, input hashes, resolved configuration,
and environment versions. It refits on all training rows for the submission.
It supports regression, classification labels, and one binary probability column.
Adapt the workflow for images, text, multi-target outputs, or other official metrics.

In [ ]:
# Edit configs/baseline.json before running:
# from src.config import Config
# from src.train import run
#
# config = Config.load(REPO_ROOT / "configs/baseline.json")
# metrics = run(config, DATA_ROOT, OUTPUT_ROOT / "runs" / "baseline-001")
# print(metrics)

## Optional accelerators

Install only the frameworks you need. Use the official PyTorch installation selector
linked in the README for your platform and hardware; keep its chosen versions in your
competition environment. Import optional libraries in the cells that use them.

`seed_everything` seeds Python and NumPy only. For PyTorch, seed its RNG separately,
set deterministic algorithms if required, and seed data-loader workers. GPU and
cross-version reproducibility require additional care.

## Submission

Build predictions in sample-submission order, then call
`src.submission.write_submission` with explicit `id_columns` and, when applicable,
`probability_columns`. It checks schema, IDs, row counts, missing/non-finite values,
and probability bounds. New files are created exclusively; overwriting is opt-in.
For multi-class probabilities, also check row sums and the competition's class order.
Validate the official metric and output semantics before uploading manually.